In [1]:
import pandas as pd
import os
import numpy as np
import librosa
import time
from sklearn.model_selection import train_test_split

In [3]:
# 데이터프레임 불러오기
fifth2nd_df = pd.read_csv(r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\5차년도_2차.csv", encoding='cp949')
fifth2nd_df

,wav_id,발화문,상황,1번 감정,1번 감정세기,2번 감정,2번 감정세기,3번 감정,3번 감정세기,4번 감정,4번감정세기,5번 감정,5번 감정세기,나이,성별
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,angry,2,surprise,2,happiness,2,happiness,2,happiness,2,48,female
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,neutral,0,happiness,2,happiness,2,happiness,2,happiness,2,48,female
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,angry,2,happiness,2,happiness,2,happiness,2,happiness,2,48,female
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,angry,2,happiness,2,happiness,2,happiness,2,happiness,1,48,female
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,happiness,2,happiness,1,happiness,2,happiness,1,happiness,1,48,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,happiness,1,sadness,1,sadness,2,sadness,1,sadness,1,23,female
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,sadness,1,fear,1,sadness,2,sadness,1,neutral,0,23,female
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,sadness,1,neutral,0,sadness,2,fear,1,sadness,1,23,female
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,disgust,1,sadness,1,neutral,0,happiness,1,sadness,1,23,female


In [7]:
print(fifth2nd_df.columns)

Index(['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정',
       '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별'],
      dtype='object')


In [9]:
# 필요없는 칼럼 제거
drop_columns = ['1번 감정','1번 감정세기','2번 감정','2번 감정세기','3번 감정','3번 감정세기','4번 감정','4번감정세기','5번 감정','5번 감정세기']
fifth2nd_df.drop(columns = drop_columns, axis=1, inplace=True)
fifth2nd_df

,wav_id,발화문,상황,나이,성별
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,48,female
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,48,female
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,48,female
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,48,female
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,48,female
...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,23,female
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,23,female
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,23,female
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,23,female


In [11]:
# 데이터프레임에 음성파일 경로 추가
AUDIO_PATH = r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\5차년도_2차"
fifth2nd_df['file_path'] = fifth2nd_df['wav_id'].apply(lambda x: os.path.join(AUDIO_PATH, x + '.wav'))
fifth2nd_df

,wav_id,발화문,상황,나이,성별,file_path
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [13]:
# csv와 음성파일 매칭시켜 음성 파일이 존재하지 않는 csv 제거
# 파일 존재여부 확인
def check_file_exists(file_path):
    if pd.isna(file_path): 
        return False
    return os.path.exists(file_path)

# file_exists column을 새로 만들어 file_path가 True인지 False인지 데이터프레임에 추가
fifth2nd_df['file_exists'] = fifth2nd_df['file_path'].apply(check_file_exists)

# file_exists가 True인 row만 선택해서 df_clean에 copy
df_clean = fifth2nd_df[fifth2nd_df['file_exists']].copy()
# file_exists column drop
df_clean.drop(columns=['file_exists'], inplace=True)

print(f"제거 전 갯수: {len(fifth2nd_df)}")
print(f"제거 후 갯수: {len(df_clean)}")
print(f"제거된 갯수: {len(fifth2nd_df) - len(df_clean)}")

제거 전 갯수: 19374
제거 후 갯수: 19374
제거된 갯수: 0


In [15]:
df_clean

,wav_id,발화문,상황,나이,성별,file_path
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,48,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,23,female,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [17]:
#상황 column을 감정으로 변경
df_clean.rename(columns={'상황':'감정'}, inplace=True)
#나이,성별 column drop => mfcc 중복 최소화
df_clean.drop(columns=['나이','성별'], inplace=True)
df_clean

,wav_id,발화문,감정,file_path
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [19]:
# MFCC : 음성 데이터를 특징 벡터화해주는 알고리즘
# file_path를 받아 특징 추출 함수 정의
def extract_features(file_path):
    try:
        # y : 음성데이터의 numpy 배열
        # sr :샘플링 속도 / sr=16000 : 초당 16000개의 샘플을 가지고 있는 데이터 / 16000인 이유 : 사람의 목소리는 대부분 16000Hz 안에 포함
        # liborsa : 음성 파일 불러오기
        y, sr = librosa.load(file_path, sr=16000) 
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None
    # n_mfcc:mfcc의 개수를 정해주는 파라미터
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40) 
    mfccs_mean = np.mean(mfccs.T, axis=0) 
    
    return mfccs_mean

print("특징 추출")
start_time = time.time()

df_clean['features'] = df_clean['file_path'].apply(extract_features)

end_time = time.time()
print(f"--- 특징 추출 완료. 총 소요 시간 : {end_time - start_time:.2f}초 ---")

# 특징 추출 중 오류(None)가 발생한 행은 제거
df_clean.dropna(subset=['features'], inplace=True)

# 최종 확인
print("\n[추출 결과 확인]")
print(df_clean.head())
print(f"처리된 데이터 수: {len(df_clean)}개")
print(f"특징 벡터 크기: {df_clean['features'].iloc[0].shape if len(df_clean) > 0 else 'N/A'}")

특징 추출


C:\Users\user\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


--- 특징 추출 완료. 총 소요 시간 : 903.59초 ---

[추출 결과 확인]
                     wav_id  \
0  5f4141e29dd513131eacee2f   
1  5f4141f59dd513131eacee30   
2  5f4142119dd513131eacee31   
3  5f4142279dd513131eacee32   
4  5f3c9ed98a3c1005aa97c4bd   

                                                 발화문         감정  \
0                                   헐! 나 이벤트에 당첨 됐어.  happiness   
1        내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.  happiness   
2                        한 명 뽑는 거였는데, 그게 바로 내가 된 거야.  happiness   
3  당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...  happiness   
4                   에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.    neutral   

                                           file_path  \
0  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
1  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
2  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
3  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
4  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   

                         

In [20]:
df_clean

,wav_id,발화문,감정,file_path,features
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-420.6026, 107.821625, -12.226244, 8.302838, ..."
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-419.99634, 97.58277, 1.5128045, 27.873543, -..."
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-394.49686, 81.0478, -13.852878, 20.84466, 0...."
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-392.5664, 85.65668, -5.396113, 23.744675, 7...."
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-357.05618, 92.17872, -10.868566, 5.9285994, ..."
...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-676.4793, 77.35785, 39.959843, 12.985315, -1..."
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-705.17285, 91.72122, 48.269196, 10.148054, 2..."
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-607.86053, 88.71402, 29.192072, 27.996576, -..."
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-654.3318, 71.630356, 39.627895, 8.481316, -7..."


In [27]:
#추출된 40개의 음성 특징을 column으로 분리
features_df = pd.DataFrame(df_clean['features'].tolist(), index=df_clean.index)
# 기존의 데이터프레임과 합치기
df_final_csv = pd.concat([df_clean.drop('features', axis=1), features_df], axis=1)
# column이름 정리
new_columns = {i: f'feature_{i}' for i in range(features_df.shape[1])}
df_final_csv.rename(columns=new_columns, inplace=True)
#csv파일로 새로 저장하기
output_file_path = "preprocessing_fifth2nd_df.csv"
df_final_csv.to_csv(output_file_path, index=False) 

In [22]:
df_final_csv

,wav_id,발화문,감정,file_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39
0,5f4141e29dd513131eacee2f,헐! 나 이벤트에 당첨 됐어.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-420.602600,107.821625,-12.226244,8.302838,-3.887772,-8.148922,...,-4.281911,-0.565309,-1.793639,-1.085530,-1.011614,-2.503790,-2.196795,-0.706601,-6.327791,-1.476724
1,5f4141f59dd513131eacee30,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-419.996338,97.582771,1.512805,27.873543,-11.392269,-14.837324,...,-3.203922,-2.023266,-4.294899,-3.810512,-3.694019,-3.952207,-3.749046,-1.531297,-6.454884,-3.241581
2,5f4142119dd513131eacee31,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-394.496857,81.047798,-13.852878,20.844660,0.986366,-17.048010,...,-5.234536,-1.518592,-2.560154,-3.000303,-2.935558,-3.822275,-4.214672,-3.392461,-5.444370,-2.317531
3,5f4142279dd513131eacee32,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-392.566406,85.656677,-5.396113,23.744675,7.186864,-16.915174,...,-2.136206,-1.810101,-3.819902,-2.279323,-4.228208,-1.570369,-4.338634,-2.479012,-5.733429,-2.569989
4,5f3c9ed98a3c1005aa97c4bd,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,neutral,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-357.056183,92.178719,-10.868566,5.928599,15.360057,-9.087642,...,-3.445714,-1.641884,-3.905374,-3.062182,-2.339470,-0.786949,-3.288026,-2.092503,-6.126595,-0.757144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19369,5fbe313c44697678c497c05a,나 엘리베이터에 갇혔어.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-676.479309,77.357849,39.959843,12.985315,-1.687219,-9.780457,...,-8.972448,-0.000537,-7.604712,-3.000915,-4.873243,-5.738294,-6.647201,-6.182743,-2.822929,-0.613307
19370,5fbe251044697678c497bfb8,하지만 기분이 나쁜 걸 어떡해?,angry,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-705.172852,91.721222,48.269196,10.148054,2.286211,-8.652617,...,-5.061596,-2.620082,-8.606046,-4.873455,-7.773482,-4.617000,-9.320764,-5.009315,-5.978668,-1.507810
19371,5fbe31584c55eb78bd7cee7f,자취방 엘리베이턴데 정전인가봐.,fear,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-607.860535,88.714020,29.192072,27.996576,-6.677667,-1.793983,...,0.787517,5.824045,-6.530383,-6.263544,-5.164907,-5.231442,-6.044603,-6.259582,-3.916384,-2.584201
19372,5fbe2f8544697678c497c047,나 드디어 프로젝트 끝났어!,happiness,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-654.331787,71.630356,39.627895,8.481316,-7.116361,-9.982390,...,-7.829257,-4.237885,-5.208100,-6.046793,-3.439856,-7.189891,-7.277931,-3.504910,-5.514778,-3.490296
